

# Laboratorio: Búsqueda Competitiva (Minimax y Alpha-Beta)

**Profesor:** M. Alania | **Curso:** Inteligencia Artificial

Bienvenidos a este entorno interactivo. En los juegos de adversarios (como Ajedrez, Go o Tres en Raya), no buscamos un "camino óptimo" estático, pues nos enfrentamos a un oponente racional que intentará minimizar nuestra utilidad.

* **Minimax:** Un algoritmo de búsqueda exhaustiva que explora todo el árbol de decisiones. Asume que nosotros (MAX) buscaremos maximizar el puntaje, mientras que el oponente (MIN) buscará minimizarlo.
* **Poda Alpha-Beta ( $\alpha-\beta$ ):** Una optimización matemática estricta sobre Minimax. Nos permite ignorar (podar) ramas del árbol que lógicamente no cambiarán nuestra decisión final, reduciendo la complejidad temporal de $O(b^m)$ al mejor caso de $O(b^{m/2})$.

**Objetivo del Laboratorio:**
Interactuar con árboles de juego dinámicos para comprender empíricamente la propagación de valores, la condición matemática de poda, y culminar con el desarrollo de una IA interactiva invencible para Tic-Tac-Toe.

---

### [Celda de Código 1]

In [ ]:
# ==========================================
# IMPORTACIÓN DE LIBRERÍAS
# ==========================================
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import math
import random

# Configuración de visualización
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente. Entorno listo.")

Librerías importadas correctamente. Entorno listo.


---

## 1. Visualización y Estructura de Árboles de Juego

Implementaremos el motor base para generar árboles binarios de juego.

**Explicación Didáctica:**
En memoria, un árbol es un grafo dirigido. Para visualizarlo matemáticamente como lo hacemos en pizarra (raíz arriba, hojas abajo), calcularemos posiciones estáticas para cada nodo. Los nodos cuadrados representarán decisiones de **MAX**, y los circulares decisiones de **MIN**.

---

### [Celda de Código 2]

In [ ]:
# ==========================================
# GENERADOR Y VISUALIZADOR DE ÁRBOLES
# ==========================================
def generate_tree(depth, branching_factor=2):
    """Genera un árbol perfecto con valores aleatorios en las hojas."""
    G = nx.DiGraph()
    leaf_values = {}

    def build_tree(node, current_depth, is_max):
        G.add_node(node, is_max=is_max, value=None, alpha=-math.inf, beta=math.inf, pruned=False)
        if current_depth == depth:
            val = random.randint(1, 20)
            G.nodes[node]['value'] = val
            leaf_values[node] = val
            return

        for i in range(branching_factor):
            child_node = f"{node}_{i}"
            G.add_edge(node, child_node)
            build_tree(child_node, current_depth + 1, not is_max)

    build_tree("R", 0, True)
    return G, leaf_values

def hierarchy_pos(G, root, width=1., vert_gap=0.2, vert_loc=0, xcenter=0.5):
    """Calcula las posiciones XY para dibujar el árbol de manera estructurada."""
    pos = {root: (xcenter, vert_loc)}
    children = list(G.successors(root))
    if not children: return pos
    dx = width / len(children)
    nextx = xcenter - width/2 - dx/2
    for child in children:
        nextx += dx
        pos.update(hierarchy_pos(G, child, width=dx, vert_gap=vert_gap,
                                 vert_loc=vert_loc-vert_gap, xcenter=nextx))
    return pos

def draw_tree(G, pos, current_node=None, message=""):
    """Dibuja el grafo G con colores basados en el estado algorítmico."""
    plt.figure(figsize=(10, 6))

    # Nodos podados vs activos
    active_nodes = [n for n in G.nodes() if not G.nodes[n].get('pruned', False)]
    pruned_nodes = [n for n in G.nodes() if G.nodes[n].get('pruned', False)]

    # MAX (Cuadrados), MIN (Círculos)
    max_nodes = [n for n in active_nodes if G.nodes[n]['is_max']]
    min_nodes = [n for n in active_nodes if not G.nodes[n]['is_max']]

    # Dibujar aristas (Rojas si están podadas)
    edges = G.edges()
    active_edges = [(u, v) for u, v in edges if not G.nodes[v].get('pruned', False)]
    pruned_edges = [(u, v) for u, v in edges if G.nodes[v].get('pruned', False)]

    nx.draw_networkx_edges(G, pos, edgelist=active_edges, arrows=False, edge_color='black')
    nx.draw_networkx_edges(G, pos, edgelist=pruned_edges, arrows=False, edge_color='red', style='dashed')

    # Colores: Resaltar el nodo actual
    color_map_max = ['yellow' if n == current_node else 'lightblue' for n in max_nodes]
    color_map_min = ['yellow' if n == current_node else 'lightgreen' for n in min_nodes]

    nx.draw_networkx_nodes(G, pos, nodelist=max_nodes, node_shape='s', node_color=color_map_max, node_size=800, edgecolors='black')
    nx.draw_networkx_nodes(G, pos, nodelist=min_nodes, node_shape='o', node_color=color_map_min, node_size=800, edgecolors='black')
    nx.draw_networkx_nodes(G, pos, nodelist=pruned_nodes, node_color='gray', node_size=400, alpha=0.5)

    # Etiquetas (Mostrar valores, y alpha/beta si existen)
    labels = {}
    for n in G.nodes():
        if G.nodes[n].get('pruned', False): continue
        val = G.nodes[n]['value']
        text = str(val) if val is not None else "?"
        # Si es un nodo de Alpha-Beta, mostrar [a, b]
        if G.nodes[n]['alpha'] != -math.inf or G.nodes[n]['beta'] != math.inf:
            a = G.nodes[n]['alpha'] if G.nodes[n]['alpha'] != -math.inf else "-∞"
            b = G.nodes[n]['beta'] if G.nodes[n]['beta'] != math.inf else "+∞"
            text += f"\n[{a},{b}]"
        labels[n] = text

    nx.draw_networkx_labels(G, pos, labels=labels, font_size=10)
    plt.title(f"Búsqueda Adversarial\n{message}", fontsize=14)
    plt.axis('off')
    plt.show()

---

## 2. Minimax y Alpha-Beta: Motores de Generación Paso a Paso

Utilizaremos "generadores" en Python (`yield`). Esto nos permite pausar el algoritmo recursivo en medio de su ejecución. Cada vez que apretemos el botón "Siguiente", extraeremos un fotograma de la evaluación.

* **Errores comunes a evitar:** Muchos estudiantes olvidan que MIN y MAX actualizan sus valores propagados de manera inversa. MAX actualiza su $\alpha$ (piso garantizado), mientras que MIN actualiza su $\beta$ (techo máximo permitido).

---

### [Celda de Código 3]

In [ ]:
# ==========================================
# ALGORITMOS CON EJECUCIÓN PASO A PASO (GENERADORES)
# ==========================================

def minimax_generator(G, node):
    """Generador paso a paso del algoritmo Minimax."""
    yield (node, f"Evaluando nodo {node}")

    children = list(G.successors(node))
    if not children:
        yield (node, f"Hoja alcanzada: Valor {G.nodes[node]['value']}")
        return G.nodes[node]['value']

    is_max = G.nodes[node]['is_max']
    best_val = -math.inf if is_max else math.inf

    for child in children:
        # Llamada recursiva usando yield from
        val = yield from minimax_generator(G, child)

        if is_max:
            if val > best_val:
                best_val = val
            yield (node, f"MAX actualiza su mejor valor a {best_val}")
        else:
            if val < best_val:
                best_val = val
            yield (node, f"MIN actualiza su mejor valor a {best_val}")

    G.nodes[node]['value'] = best_val
    yield (node, f"Valor final de {node} determinado como {best_val}")
    return best_val

def alphabeta_generator(G, node, alpha, beta):
    """Generador paso a paso del algoritmo Alpha-Beta Pruning."""
    G.nodes[node]['alpha'] = alpha
    G.nodes[node]['beta'] = beta
    yield (node, f"Visitando {node}. α={alpha}, β={beta}")

    children = list(G.successors(node))
    if not children:
        yield (node, f"Hoja evaluada: {G.nodes[node]['value']}")
        return G.nodes[node]['value']

    is_max = G.nodes[node]['is_max']
    best_val = -math.inf if is_max else math.inf

    for i, child in enumerate(children):
        val = yield from alphabeta_generator(G, child, alpha, beta)

        if is_max:
            best_val = max(best_val, val)
            G.nodes[node]['value'] = best_val
            alpha = max(alpha, best_val)
            G.nodes[node]['alpha'] = alpha
            yield (node, f"MAX actualiza: V={best_val}, α={alpha}")
            if best_val >= beta:
                yield (node, f"¡PODA! v ({best_val}) >= β ({beta}). Se ignoran hermanos restantes.")
                # Marcar ramas restantes como podadas
                for prune_child in children[i+1:]:
                    mark_pruned(G, prune_child)
                break # Romper el ciclo es la Poda real
        else:
            best_val = min(best_val, val)
            G.nodes[node]['value'] = best_val
            beta = min(beta, best_val)
            G.nodes[node]['beta'] = beta
            yield (node, f"MIN actualiza: V={best_val}, β={beta}")
            if best_val <= alpha:
                yield (node, f"¡PODA! v ({best_val}) <= α ({alpha}). Se ignoran hermanos restantes.")
                for prune_child in children[i+1:]:
                    mark_pruned(G, prune_child)
                break

    yield (node, f"Retornando {best_val} desde {node}")
    return best_val

def mark_pruned(G, node):
    """Función auxiliar para marcar recursivamente un subárbol como podado visualmente."""
    G.nodes[node]['pruned'] = True
    for child in G.successors(node):
        mark_pruned(G, child)

---

## 3. Panel de Exploración Interactiva

A continuación, ejecute la celda para desplegar el Panel de Control.
**Instrucciones Pedagógicas:**

1. Seleccione "Minimax" y de clic en "Siguiente Paso" repetidamente. Observe cómo la búsqueda desciende hasta el final de la rama izquierda primero (Depth-First Search).
2. Seleccione "Alpha-Beta". Revise la propagación de los corchetes `[alpha, beta]`.
3. **Poda (Color Gris/Rojo):** Note cómo, cuando el valor garantiza una pérdida o irrelevancia para el jugador de niveles superiores, se corta la exploración.
4. **Análisis Crítico:** Cambie el "Orden de Hojas". Si ordena descendentemente, MAX evalúa lo mejor primero y MIN lo peor. ¿Qué impacto tiene en la cantidad de podas?

---

### [Celda de Código 4]

In [ ]:
# ==========================================
# INTERFAZ GRÁFICA INTERACTIVA CON IPYWIDGETS
# ==========================================
class TreeSimulator:
    def __init__(self):
        self.output = widgets.Output()
        self.G = None
        self.pos = None
        self.generator = None
        self.nodes_visited = 0
        self.setup_ui()

    def setup_ui(self):
        style = {'description_width': 'initial'}
        self.algo_dropdown = widgets.Dropdown(options=['Minimax', 'Alpha-Beta'], value='Alpha-Beta', description='Algoritmo:')
        self.depth_slider = widgets.IntSlider(value=3, min=2, max=4, description='Profundidad:')
        self.branch_slider = widgets.IntSlider(value=2, min=2, max=3, description='Ramificación:')
        self.order_dropdown = widgets.Dropdown(options=['Aleatorio', 'Óptimo (MAX)', 'Peor Caso'], description='Orden Hojas:', style=style)

        self.btn_gen = widgets.Button(description='Generar Árbol', button_style='primary')
        self.btn_step = widgets.Button(description='Siguiente Paso', button_style='success', disabled=True)
        self.btn_run = widgets.Button(description='Ejecutar Rápido', button_style='info', disabled=True)
        self.label_info = widgets.Label(value="Configura y genera un árbol.")

        self.btn_gen.on_click(self.generate_new_tree)
        self.btn_step.on_click(self.next_step)
        self.btn_run.on_click(self.run_all)

        controls = widgets.VBox([
            widgets.HBox([self.depth_slider, self.branch_slider, self.order_dropdown]),
            widgets.HBox([self.algo_dropdown, self.btn_gen, self.btn_step, self.btn_run]),
            self.label_info
        ])
        display(controls, self.output)

    def generate_new_tree(self, b):
        self.btn_step.disabled = False
        self.btn_run.disabled = False
        self.nodes_visited = 0
        self.G, leaves = generate_tree(self.depth_slider.value, self.branch_slider.value)
        self.pos = hierarchy_pos(self.G, "R")

        # Alterar orden de las hojas para simulaciones de mejor/peor caso
        if self.order_dropdown.value != 'Aleatorio':
            sorted_vals = sorted(leaves.values(), reverse=(self.order_dropdown.value == 'Óptimo (MAX)'))
            for i, node in enumerate(leaves.keys()):
                self.G.nodes[node]['value'] = sorted_vals[i]

        # Reiniciar generador
        if self.algo_dropdown.value == 'Minimax':
            self.generator = minimax_generator(self.G, "R")
        else:
            self.generator = alphabeta_generator(self.G, "R", -math.inf, math.inf)

        self.update_plot(None, "Árbol inicializado. Presiona 'Siguiente Paso'.")

    def next_step(self, b=None):
        try:
            node, message = next(self.generator)
            self.nodes_visited += 1
            self.label_info.value = f"Nodos procesados: {self.nodes_visited} | Log: {message}"
            self.update_plot(node, message)
        except StopIteration as e:
            final_val = e.value
            self.btn_step.disabled = True
            self.btn_run.disabled = True
            self.label_info.value = f"¡Búsqueda Finalizada! Valor de la Raíz: {final_val}. Total pasos: {self.nodes_visited}"
            self.update_plot("R", f"Terminado. Resultado Óptimo = {final_val}")

    def run_all(self, b):
        self.btn_step.disabled = True
        self.btn_run.disabled = True
        try:
            while True:
                node, message = next(self.generator)
                self.nodes_visited += 1
        except StopIteration as e:
            self.label_info.value = f"¡Ejecución Rápida Finalizada! Valor de la Raíz: {e.value}. Total pasos: {self.nodes_visited}"
            self.update_plot("R", f"Terminado. Resultado Óptimo = {e.value}")

    def update_plot(self, current_node, message):
        with self.output:
            clear_output(wait=True)
            draw_tree(self.G, self.pos, current_node, message)

# Instanciar el entorno interactivo
TreeSimulator()

Output()

---

## 4. Implementación Práctica: IA en Tic-Tac-Toe

Para consolidar la teoría, programaremos el clásico juego de Tres en Raya.
Aquí, el árbol de estados no es aleatorio; se genera dinámicamente según las reglas del juego.

* **Estado (Board):** Una lista de 9 posiciones.
* **Evaluación:** Gana X = +10, Gana O = -10, Empate = 0.
* **Heurística de Profundidad:** A la evaluación le restaremos la profundidad, para que la IA prefiera ganar rápido en 2 jugadas en lugar de ganar lentamente en 6.

---

### [Celda de Código 5]

In [ ]:
# ==========================================
# MOTOR DEL JUEGO TIC-TAC-TOE CON IA
# ==========================================

def check_winner(board):
    win_states = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
    for a, b, c in win_states:
        if board[a] == board[b] == board[c] and board[a] != ' ':
            return board[a]
    if ' ' not in board: return 'Draw'
    return None

def minimax_ttt(board, depth, is_max, use_ab=True, alpha=-math.inf, beta=math.inf):
    """Evalúa el estado del tablero y devuelve (Puntaje)."""
    winner = check_winner(board)
    if winner == 'X': return 10 - depth  # IA prefiere ganar rápido
    if winner == 'O': return -10 + depth # IA prefiere que humano gane tarde
    if winner == 'Draw': return 0

    if is_max: # Turno de IA (X)
        best_score = -math.inf
        for i in range(9):
            if board[i] == ' ':
                board[i] = 'X'
                score = minimax_ttt(board, depth + 1, False, use_ab, alpha, beta)
                board[i] = ' '
                best_score = max(score, best_score)
                if use_ab:
                    alpha = max(alpha, best_score)
                    if best_score >= beta: break # Poda
        return best_score
    else: # Turno del Humano (O)
        best_score = math.inf
        for i in range(9):
            if board[i] == ' ':
                board[i] = 'O'
                score = minimax_ttt(board, depth + 1, True, use_ab, alpha, beta)
                board[i] = ' '
                best_score = min(score, best_score)
                if use_ab:
                    beta = min(beta, best_score)
                    if best_score <= alpha: break # Poda
        return best_score

def get_best_move(board, use_ab=True):
    """Inicia la búsqueda en el nodo raíz y recolecta métricas."""
    best_score = -math.inf
    best_move = None
    start_time = time.time()

    # Análisis de Alternativas (Visualización del proceso de decisión)
    alternatives = []

    for i in range(9):
        if board[i] == ' ':
            board[i] = 'X'
            score = minimax_ttt(board, 0, False, use_ab)
            board[i] = ' '
            alternatives.append((i, score))
            if score > best_score:
                best_score = score
                best_move = i

    elapsed = time.time() - start_time
    return best_move, elapsed, alternatives

---

## 5. ¡Enfréntate a la Inteligencia Artificial!

Juega contra la implementación que acabas de ver.
**Experimentación Didáctica:**

1. Al desactivar Alpha-Beta, notarás que el juego sigue jugando perfecto, pero el tiempo de cálculo interno (aunque milisegundos en Tic-Tac-Toe) es mayor comparativamente. Alpha-Beta explora menos de la mitad del espacio total ($9! = 362,880$ estados máximos limitados por victorias previas).
2. Observa en la consola inferior el "Proceso de Decisión" de la IA: verás qué puntaje le asigna a cada jugada posible. Un puntaje positivo indica que la IA ha encontrado una secuencia de victoria forzada. Un puntaje 0 indica que, si juegas perfecto, solo puedes aspirar a empatarle. ¡Intenta ganar (alerta de spoiler: matemáticamente imposible)!

---

### [Celda de Código 6]

In [ ]:
# ==========================================
# INTERFAZ INTERACTIVA PARA JUGAR CONTRA LA IA
# ==========================================

class TicTacToeGame:
    def __init__(self):
        self.board = [' '] * 9
        self.buttons = []
        self.output = widgets.Output()
        self.setup_ui()

    def setup_ui(self):
        self.algo_toggle = widgets.Checkbox(value=True, description='Usar Poda Alpha-Beta')
        self.reset_btn = widgets.Button(description='Reiniciar Juego', button_style='danger')
        self.reset_btn.on_click(self.reset_game)

        grid_layout = widgets.Layout(grid_template_columns="repeat(3, 80px)", grid_gap="5px")

        for i in range(9):
            btn = widgets.Button(description=' ', layout=widgets.Layout(width='80px', height='80px'))
            btn.style.button_color = 'white'
            btn.style.font_weight = 'bold'
            btn.on_click(self.handle_click)
            self.buttons.append(btn)

        grid = widgets.GridBox(self.buttons, layout=grid_layout)

        controls = widgets.VBox([
            widgets.HTML("<h3>Tú eres 'O'. La IA es 'X'. Empiezas tú.</h3>"),
            self.algo_toggle,
            grid,
            self.reset_btn
        ])

        display(controls, self.output)

    def reset_game(self, b=None):
        self.board = [' '] * 9
        for btn in self.buttons:
            btn.description = ' '
            btn.disabled = False
            btn.style.button_color = 'white'
        with self.output:
            clear_output()
            print("Juego reiniciado. Tu turno.")

    def handle_click(self, b):
        idx = self.buttons.index(b)
        if self.board[idx] != ' ': return

        # Movimiento del humano
        self.board[idx] = 'O'
        b.description = 'O'
        b.style.button_color = 'lightblue'
        b.disabled = True

        if self.check_game_over(): return

        # Movimiento de la IA
        with self.output:
            clear_output()
            print("La IA está calculando su jugada...")

        move, t_elapsed, alts = get_best_move(self.board, use_ab=self.algo_toggle.value)

        self.board[move] = 'X'
        self.buttons[move].description = 'X'
        self.buttons[move].style.button_color = 'salmon'
        self.buttons[move].disabled = True

        with self.output:
            clear_output()
            print(f"La IA eligió la posición {move} en {t_elapsed:.4f} segundos.")
            print("\n--- ANÁLISIS DEL PROCESO DE DECISIÓN (Raíz) ---")
            print("Alternativas evaluadas por la IA (Posición -> Puntaje Proyectado):")
            for m, s in alts:
                status = "Gana IA forzado" if s > 0 else ("Empate forzado" if s == 0 else "Pierde IA")
                print(f"Posición {m}: Puntaje {s} ({status})")

        self.check_game_over()

    def check_game_over(self):
        winner = check_winner(self.board)
        if winner:
            for btn in self.buttons: btn.disabled = True
            with self.output:
                print("---------------------------------")
                if winner == 'Draw': print("¡ES UN EMPATE! Jugaste a la perfección.")
                elif winner == 'X': print("¡LA IA GANA! El algoritmo es invencible.")
                else: print("¡TÚ GANAS! (Esto no debería ocurrir matemáticamente...)")
            return True
        return False

# Iniciar juego
TicTacToeGame()

Output()

---

## 6. Conclusiones Extensibles y Trabajo Adicional

Has visualizado cómo los algoritmos navegan y cómo se podan subárboles matemáticamente redundantes.

**¿Cómo podemos escalar esto a juegos como Ajedrez?**
El Tic-Tac-Toe es suficientemente pequeño para evaluarse hasta los nodos terminales reales. En ajedrez, esto tomaría millones de años. Para escalarlo necesitarías:

1. **Límite de Profundidad (Depth Limit):** Detener la recursión en `depth == 5`, por ejemplo.
2. **Función de Evaluación Heurística:** Una función que no determine "quién ganó", sino "quién tiene ventaja" (ej. contar piezas del tablero, control del centro). Se llamaría cuando `depth` llegue a su límite.
3. **Ordenamiento de Jugadas (Move Ordering):** Como observaste en la visualización, la poda depende críticamente de revisar las mejores jugadas primero. Evaluar capturas y jaques antes que movimientos pasivos aumenta dramáticamente la cantidad de Ramas Alpha-Beta cortadas.